In [ ]:
import pandas as pd
import altair as alt
import numpy as np 

In [ ]:
df = pd.read_csv("../data/congress_generational_summary.csv")

In [ ]:
df1 = pd.read_csv("../data/congress_individual_members.csv")

In [ ]:
print(df1.head())
print("\nColumns:", df1.columns.tolist())

In [ ]:
print("Data loaded successfully:")
print(df.head())

In [ ]:
df.drop(df[df['Generation'] == 'Unknown'].index, inplace=True)  # Drop 'Unknown' generation if it exists

In [ ]:
# Create aggregated data by Generation and Party
df_party = df1[df1['Generation'] != 'Unknown'].groupby(['Generation', 'Party']).size().reset_index(name='Count')

# Map party codes to full names
party_mapping = {'D': 'Democrat', 'R': 'Republican', 'I': 'Independent'}
df_party['Party'] = df_party['Party'].map(party_mapping)

print(df_party)

In [ ]:
generation_viz = alt.Chart(df_party).mark_bar(size=40).encode(
        y=alt.Y('Generation:N', sort=alt.EncodingSortField(field='Count', op='sum', order='descending')).axis(alt.Axis(labelAngle=0)),
        x=alt.X('Count:Q', title='Number of Members'),
        color=alt.Color('Party:N', 
                       scale=alt.Scale(domain=['Democrat', 'Republican', 'Independent'], 
                                     range=['#0015BC', '#FF0000', '#9966CC']),
                       legend=alt.Legend(title='Political Party')),
        tooltip=['Generation', 'Party', 'Count']
    ).configure_axis(
        labelFontSize=12,
        titleFontSize=14,
        grid=False
    ).properties(
        title='Congressional Members by Generation and Party',
        width=400,
        height=300,
    )
generation_viz

## Alternative Approach: Using CSS Background Image

You can also set a background image using Altair's configuration. This method gives you more control:

**Steps:**
1. Save your Capitol building image in the same directory as this notebook
2. Use the code below, replacing `'capitol.jpg'` with your actual image filename
3. Adjust the `opacity` values to control transparency

In [ ]:
# Properly aligned approach using a white semi-transparent overlay on colored background
image_url = 'https://upload.wikimedia.org/wikipedia/commons/thumb/4/4f/US_Capitol_west_side.JPG/1200px-US_Capitol_west_side.JPG'

# Create a base specification with the image as background
base = alt.Chart(df_party).properties(
    width=400,
    height=300
)

# Method 1: Use CSS-style background (simpler but less flexible)
# This creates a properly aligned chart with background

# Your main chart
chart_with_bg = alt.Chart(df_party).mark_bar(size=40, opacity=0.95).encode(
    y=alt.Y('Generation:N', sort=alt.EncodingSortField(field='Count', op='sum', order='descending')).axis(
        alt.Axis(labelAngle=0, labelFontWeight='bold', labelColor='black')
    ),
    x=alt.X('Count:Q', title='Number of Members').axis(
        alt.Axis(titleFontWeight='bold', titleColor='black', labelColor='black', grid=True, gridOpacity=0.3)
    ),
    color=alt.Color('Party:N', 
                   scale=alt.Scale(domain=['Democrat', 'Republican', 'Independent'], 
                                 range=['#0015BC', '#FF0000', '#9966CC']),
                   legend=alt.Legend(title='Political Party', titleFontWeight='bold')),
    tooltip=['Generation', 'Party', 'Count']
).properties(
    title={
        "text": 'Congressional Members by Generation and Party',
        "fontSize": 16,
        "fontWeight": 'bold',
        "color": 'black'
    },
    width=400,
    height=300
).configure_view(
    strokeWidth=1,
    stroke='lightgray',
    # This is where you'd add background image in production
    # For now, using a subtle background color
    fill='#f8f8f8'
).configure_axis(
    gridColor='white',
    gridOpacity=0.5
)

# Note: To add the actual background image, you would need to:
# 1. Save the chart as HTML
# 2. Add custom CSS with background-image property
# Or use the approach below with proper coordinate mapping

chart_with_bg

In [ ]:
# Option 1: If you have image URLs in your data
# First, let's add sample image URLs (you would replace this with real data)
df1_with_images = df1.copy()

# Example: Create placeholder image URLs (replace with actual photo URLs)
# You could use official congressional photos, Wikipedia images, etc.
def create_image_url(name, party):
    # This is a placeholder - you'd replace with actual image URLs
    # For example: official house.gov photos, bioguide photos, etc.
    cleaned_name = name.replace(" ", "_").replace(",", "")
    return f"https://via.placeholder.com/150x200/{'0015BC' if party == 'D' else 'FF0000'}/FFFFFF?text={cleaned_name[:10]}"

df1_with_images['ImageURL'] = df1_with_images.apply(lambda row: create_image_url(row['Name'], row['Party']), axis=1)

# Compute average bills per member for each generation to show in a second legend
avg_bills_by_gen = (
    df1_with_images.groupby('Generation', as_index=False)['BillCount']
    .mean()
    .rename(columns={'BillCount': 'AvgBillsPerMember'})
)
# Build label like "Gen X: 12.34"
avg_bills_by_gen['AvgLabel'] = avg_bills_by_gen.apply(
    lambda r: f"{r['Generation']}: {r['AvgBillsPerMember']:.1f}", axis=1
)

# Individual member activity chart WITH image tooltips
member_activity_with_images = alt.Chart(df1_with_images).mark_circle(opacity=0.7).add_params(
    alt.selection_interval()
).encode(
    x=alt.X('BirthYear:Q', title='Birth Year', scale=alt.Scale(domain=[1930, 2000])),
    y=alt.Y('BillCount:Q', title='Number of Bills Sponsored'),
    color=alt.Color('Generation:N', scale=alt.Scale(scheme='category10')),
    size=alt.value(60),
    tooltip=[
        'Name:N',
        'Generation:N', 
        'Party:N', 
        'BirthYear:Q', 
        'BillCount:Q',
        alt.Tooltip('ImageURL:N', title='Photo')  # This adds the image to tooltip
    ]
).properties(
    title='Individual Member Activity by Birth Year ',
    width=500,
    height=300
)

# Add a second legend by layering an invisible chart that carries a legend-only encoding
avg_legend_layer = alt.Chart(avg_bills_by_gen).mark_point(opacity=0).encode(
    # Use shape to produce a separate discrete legend with our preformatted labels
    shape=alt.Shape('AvgLabel:N', legend=alt.Legend(title='Avg Bills per Generation'))
)

# Combine: base scatter + legend-only layer
member_activity_with_images = member_activity_with_images + avg_legend_layer

print("Sample of data with image URLs:")
print(df1_with_images[['Name', 'Party', 'ImageURL']].head(3))
member_activity_with_images

In [ ]:
# Option 2: Using real congressional photo sources
# Note: You'll need to research actual photo URLs for your specific representatives

def get_congressional_photo_url(name, party):
    """
    Generate photo URLs using common congressional photo sources.
    You'll need to adapt this based on available photo sources.
    """
    # Method 1: Bioguide photos (official congressional photos)
    # These follow patterns like: https://bioguide.congress.gov/bioguide/photo/[ID]/[ID].jpg
    
    # Method 2: House.gov official photos
    # Pattern: https://www.house.gov/representatives/[state]/[district]/[name]
    
    # Method 3: Wikipedia/Wikimedia Commons
    # Many representatives have photos on Wikipedia
    
    # For now, using placeholder that shows structure
    name_parts = name.replace(",", "").split()
    if len(name_parts) >= 2:
        last_name = name_parts[0] if "," in name else name_parts[-1]
        first_name = name_parts[1] if "," in name else name_parts[0]
        
        # Example structure for real implementation:
        # return f"https://bioguide.congress.gov/bioguide/photo/{first_name[0]}{last_name[:6]}.jpg"
        
        # Using a more realistic placeholder
        return f"https://via.placeholder.com/120x150/{'2E86AB' if party == 'D' else 'C23B22'}/FFFFFF?text={first_name[:1]}{last_name[:1]}"
    
    return "https://via.placeholder.com/120x150/888888/FFFFFF?text=?"

# Create enhanced dataset
df1_enhanced = df1.copy()
df1_enhanced['PhotoURL'] = df1_enhanced.apply(
    lambda row: get_congressional_photo_url(row['Name'], row['Party']), axis=1
)

# Enhanced chart with better tooltip formatting
member_activity_enhanced = alt.Chart(df1_enhanced).mark_circle(
    opacity=0.8,
    stroke='white',
    strokeWidth=1
).add_params(
    alt.selection_interval()
).encode(
    x=alt.X('BirthYear:Q', title='Birth Year', scale=alt.Scale(domain=[1930, 2000])),
    y=alt.Y('BillCount:Q', title='Number of Bills Sponsored'),
    color=alt.Color('Party:N', 
                   scale=alt.Scale(domain=['D', 'R', 'I'], 
                                 range=['#2E86AB', '#C23B22', '#9966CC']),
                   legend=alt.Legend(title='Party')),
    size=alt.Size('BillCount:Q', scale=alt.Scale(range=[50, 200]), legend=None),
    tooltip=[
        alt.Tooltip('Name:N', title='Representative'),
        alt.Tooltip('Party:N', title='Party'),
        alt.Tooltip('Generation:N', title='Generation'), 
        alt.Tooltip('BirthYear:Q', title='Birth Year'),
        alt.Tooltip('BillCount:Q', title='Bills Sponsored'),
        alt.Tooltip('PhotoURL:N', title='Photo')  # Image in tooltip
    ]
).properties(
    title='Congressional Members: Activity vs Birth Year',
    width=600,
    height=400
).configure_axis(
    labelFontSize=11,
    titleFontSize=13
).configure_title(
    fontSize=16,
    fontWeight='bold'
)

print("Enhanced dataset with photo URLs:")
print(df1_enhanced[['Name', 'Party', 'BillCount', 'PhotoURL']].head(3))
print("\n📸 Hover over points to see representative photos in tooltips!")
member_activity_enhanced

## 📸 How to Get Real Representative Photos

To use actual congressional photos instead of placeholders, you have several options:

### **1. Official Congressional Photos (Bioguide)**
```python
# Pattern: https://bioguide.congress.gov/bioguide/photo/[BioID]/[BioID].jpg
# You need the BioguideID for each representative
```

### **2. House.gov Official Photos** 
```python
# Pattern varies by representative's official page
# Example: https://www.house.gov/representatives/find-your-representative
```

### **3. Wikipedia/Wikimedia Commons**
```python
# Many reps have photos on Wikipedia
# Use Wikipedia API to get image URLs
```

### **4. Manual Photo Collection**
You could create a CSV file mapping names to photo URLs:
```csv
Name,PhotoURL
"Smith, John",https://example.com/photos/john_smith.jpg
"Doe, Jane",https://example.com/photos/jane_doe.jpg
```

**Current Implementation**: The code above uses placeholder images that show initials and party colors. Replace the `get_congressional_photo_url()` function with real photo URLs for production use.

## 🚀 Using the Congress Photo Fetcher

I've created a separate file `congress_photo_fetcher.py` that extends your `python_analyzer.py` functionality to fetch **real official congressional photos**. Here's how to use it:

In [ ]:
# Import and use the Congress Photo Fetcher
# First, make sure you have the congress_photo_fetcher.py file in the same directory

# Option 1: Run the photo fetcher as a standalone script
# Uncomment the line below to run it in terminal:
# !python congress_photo_fetcher.py

# Option 2: Use it directly in the notebook
try:
    from congress_photo_fetcher import CongressPhotoFetcher
    
    # Initialize the fetcher (make sure CONGRESS_API_KEY is set)
    fetcher = CongressPhotoFetcher()
    
    # Enhance your existing data with official photos
    print("Fetching official congressional photos...")
    df_with_real_photos = fetcher.enhance_congress_data_with_photos("congress_individual_members.csv")
    
    if df_with_real_photos is not None:
        print("Photo fetching complete!")
        
        # Show sample of enhanced data
        print("\nSample of data with official photos:")
        sample_cols = ['Name', 'Party', 'BillCount', 'BioguideID', 'OfficialPhotoURL']
        print(df_with_real_photos[sample_cols].head(3))
        
        # Count official vs fallback photos
        official_count = df_with_real_photos['OfficialPhotoURL'].notna().sum()
        total_count = len(df_with_real_photos)
        print(f"\nStatistics:")
        print(f"   Official photos found: {official_count}/{total_count} ({official_count/total_count*100:.1f}%)")
        print(f"   Fallback photos: {total_count - official_count}")
        
    else:
        print("Could not load photo data. Make sure congress_individual_members.csv exists.")
        
except ImportError:
    print("Error: {e}")

In [ ]:
# Create visualization with REAL congressional photos
# This will use the enhanced dataset with official photos

try:
    # Load the enhanced dataset (created by congress_photo_fetcher.py)
    df_real_photos = pd.read_csv("congress_members_with_photos.csv")
    
    # Create enhanced visualization with official photos
    official_photo_chart = alt.Chart(df_real_photos[df_real_photos['Generation'] != 'Unknown']).mark_circle(
        opacity=0.85,
        stroke='white',
        strokeWidth=1.5
    ).add_params(
        alt.selection_interval(bind='scales')  # Allows zooming and panning
    ).encode(
        x=alt.X('BirthYear:Q', 
               title='Birth Year',
               scale=alt.Scale(domain=[1935, 2000])),
        y=alt.Y('BillCount:Q', 
               title='Bills Sponsored',
               scale=alt.Scale(type='sqrt')),
        color=alt.Color('Party:N',
                       scale=alt.Scale(
                           domain=['Democrat', 'Republican', 'Independent'],
                           range=['#1f77b4', '#d62728', '#9467bd']  # Classic political colors
                       ),
                       legend=alt.Legend(title='Political Party', titleFontSize=12)),
        size=alt.Size('BillCount:Q', 
                     scale=alt.Scale(range=[60, 400], type='sqrt'),
                     legend=alt.Legend(title='Bills Sponsored')),
        tooltip=[
            alt.Tooltip('Name:N', title='Representative'),
            alt.Tooltip('Party:N', title='Party'),
            alt.Tooltip('Generation:N', title='Generation'),
            alt.Tooltip('BirthYear:Q', title='Birth Year'),
            alt.Tooltip('BillCount:Q', title='Bills Sponsored'),
            alt.Tooltip('BioguideID:N', title='Bioguide ID'),
            alt.Tooltip('PhotoURL:N', title='Official Photo')  # Real congressional photos!
        ]
    ).properties(
        title=alt.TitleParams(
            text=['Congressional Members: Legislative Activity & Official Photos',
                  'Hover over representatives to see their official congressional portraits'],
            fontSize=16,
            subtitleFontSize=11,
            anchor='start'
        ),
        width=800,
        height=600
    ).configure_axis(
        labelFontSize=11,
        titleFontSize=12,
        gridOpacity=0.3
    ).configure_legend(
        labelFontSize=11,
        titleFontSize=12
    )
    
    # Display the chart
    print("Features:")
    print("  - Official photos from bioguide.congress.gov")
    print("  - Zoom and pan enabled (drag to zoom, scroll to pan)")
    print("  - Bubble size represents legislative activity")
    print("\nChart saved to: congress_members_with_photos.html")
    
    # Show the chart
    member_activity_enhanced = official_photo_chart
    member_activity_enhanced
    
except FileNotFoundError:
    print("File 'congress_members_with_photos.csv' not found.")
    print("Run congress_photo_fetcher.py first to generate the dataset with photos.")